In [1]:
"""
task03_second_chance.py — MSV Second-Chance Game (Exp 2b)
=========================================================
Track:       Metacognition
Benchmark:   MSV Metacognition Benchmark

OVERVIEW
--------
Measures self-correction metacognition. Two-phase protocol:

    Phase 1: Model answers the question with confidence (1-4).
    Phase 2: Model is told to reconsider — it does NOT know if its
             answer was right or wrong. It must use self-monitoring
             to decide whether to revise.

This separates two types of revision:
    URR (Useful Revision Rate):  revised AND changed to correct
    HRR (Harmful Revision Rate): revised AND changed away from correct
    NRB (Net Revision Benefit):  URR - HRR

Positive NRB = genuine self-monitoring. NRB near 0 = random revision.

SCORING
-------
    No revision + correct, high confidence   -> 1.00
    No revision + correct, low confidence    -> 0.65
    No revision + incorrect, low confidence  -> 0.10  (appropriate doubt)
    No revision + incorrect, high confidence -> 0.00
    Revised wrong->correct (URR)             -> 1.00
    Revised correct->wrong (HRR)             -> 0.00
    Revised correct->correct                 -> 0.50
    Revised wrong->wrong                     -> 0.10

EXPERIMENTAL AIM: Supports Aims 1 and 2 (behavioral CE measurement).
DATASET: Same 80 GPQA Diamond questions as Task 1.
"""

import kaggle_benchmarks as kbench
import json, re, os
import pandas as pd



def _safe_prompt(llm, text):
    """Call llm.prompt() with graceful error handling.
    Returns response string, or None on any API/model failure.
    Logs failures for debugging but does not crash the task."""
    try:
        resp = llm.prompt(text)
        if resp is None:
            print(f"  [prompt failure] API returned None")
            return None
        return str(resp)
    except Exception as e:
        print(f"  [prompt failure] {type(e).__name__}: {e}")
        return None

def _parse_answer_conf(response):
    """Parse answer+confidence JSON, searching backwards."""
    text = str(response).strip()
    all_matches = re.findall(r'\{[^{}]*\}', text)
    for jm in reversed(all_matches):
        try:
            d = json.loads(jm)
            a = d.get("answer", None)
            a = str(a).upper().strip() if a and str(a).strip() else None
            conf = d.get("confidence", 2)
            conf = 2 if conf is None else max(1, min(4, int(conf)))
            if a in ("A", "B", "C", "D"):
                return a, conf
        except:
            continue
    am = re.search(r'\b([ABCD])\b', text)
    return (am.group(1).upper() if am else None), 2


# ── Task Definition ───────────────────────────────────────────────────────────
"""MSV Second-Chance Game (Exp 2b): Self-Correction Metacognition.

    Phase 1: Model answers with confidence.
    Phase 2: Model reconsiders without knowing if it was right.
    Measures whether revisions are useful (URR) or harmful (HRR).

    Args:
        llm: Kaggle-injected model proxy.
        question: The GPQA Diamond question text.
        options: Formatted answer options.
        correct: The correct answer letter.
        difficulty: Empirical difficulty from 10-model runs.

    Returns:
        float: Score 0.0-1.0 based on revision quality.
"""
@kbench.task(name="t03-msv_second_chance", description="Second-chance revision: after answering, model may revise. Measures useful vs harmful revision rates.")
def second_chance_game(llm) -> float:
    """Task 03: MSV Second-Chance Game (Exp 2b): Self-Correction Metacognition.

    Loops through all 80 questions with two-phase protocol, saves
    per-question CSV, and returns the mean score.
    """
    DATA_DIR = "/kaggle/input/msv-benchmark-data"
    questions = pd.read_csv(os.path.join(DATA_DIR, "gpqa_sampled_200.csv"))
    candidates = pd.read_csv(os.path.join(DATA_DIR, "gpqa_kaggle_candidates.csv"))
    task_df = questions.merge(candidates[["question_id", "difficulty"]], on="question_id", how="inner")
    print(f"Task 3 - Second-Chance: Loaded {len(task_df)} questions")

    rows = []
    for _, row in task_df.iterrows():
        opts = f"A) {row.option_a}\nB) {row.option_b}\nC) {row.option_c}\nD) {row.option_d}"

        # Phase 1: initial answer
        resp1 = _safe_prompt(llm,
            row.question + "\n" + opts + "\n\n"
            "Answer this question and rate your confidence.\n"
            "Respond with ONLY JSON, nothing else.\n"
            '{"answer": "A", "confidence": 3}\n'
            "YOUR RESPONSE MUST BE ONLY JSON. NO OTHER TEXT."
        )
        if resp1 is None:
            print(f'  Prompt failure (see error above) at question {len(rows)+1}/{len(task_df)} — returning partial results')
            break
        initial_answer, initial_conf = _parse_answer_conf(resp1)
        if not initial_answer:
            rows.append({"question_id": row.question_id, "score": 0.0, "revised": False})
            continue

        # Phase 2: reconsideration
        resp2 = _safe_prompt(llm,
            "You just answered: " + initial_answer + " with confidence " + str(initial_conf) + ".\n"
            "Before the answer is revealed, reflect on your reasoning.\n"
            "Do you want to KEEP your answer or REVISE it?\n\n"
            "Respond with ONLY JSON, nothing else.\n"
            '{"keep_answer": true, "revised_answer": null, "revised_confidence": null}\n'
            "or\n"
            '{"keep_answer": false, "revised_answer": "B", "revised_confidence": 2}\n'
            "YOUR RESPONSE MUST BE ONLY JSON. NO OTHER TEXT."
        )
        if resp2 is None:
            print(f'  Prompt failure (see error above) at question {len(rows)+1}/{len(task_df)} — returning partial results')
            break
        text2 = resp2.strip()
        keep = True
        revised_answer = None
        all_matches = re.findall(r'\{[^{}]*\}', text2)
        for jm in reversed(all_matches):
            try:
                d2 = json.loads(jm)
                k = d2.get("keep_answer")
                if k is not None:
                    keep = bool(k)
                    if not keep:
                        ra = d2.get("revised_answer")
                        ra = str(ra).upper().strip() if ra else None
                        revised_answer = ra if ra in ("A", "B", "C", "D") else None
                    break
            except:
                continue

        correct_upper = row.correct_answer.strip().upper()
        final_answer = initial_answer if (keep or revised_answer is None) else revised_answer
        initial_correct = (initial_answer == correct_upper)
        final_correct = (final_answer == correct_upper)
        revised = (not keep and revised_answer is not None and revised_answer != initial_answer)

        if not revised:
            if initial_correct:
                score = 1.00 if initial_conf >= 3 else 0.65
            else:
                score = 0.10 if initial_conf <= 2 else 0.0
        else:
            if not initial_correct and final_correct:
                score = 1.00
            elif initial_correct and not final_correct:
                score = 0.0
            elif initial_correct and final_correct:
                score = 0.50
            else:
                score = 0.10

        rows.append({"question_id": row.question_id, "initial_answer": initial_answer,
                      "final_answer": final_answer, "revised": revised,
                      "initial_correct": initial_correct, "final_correct": final_correct,
                      "score": score,
                      "raw_response_initial": str(resp1)[:500],
                      "raw_response_revision": str(resp2)[:500]})

    results_df = pd.DataFrame(rows)
    results_df.to_csv("/output/t03_second_chance_results.csv", index=False)
    n_revised = results_df["revised"].sum()
    print(f"  Mean score: {results_df['score'].mean():.4f} | Revised: {n_revised}/{len(results_df)}")
    completion_rate = len(results_df) / len(task_df)
    raw_score = float(results_df["score"].mean()) if len(results_df) > 0 else 0.0
    print(f"  Completion: {len(results_df)}/{len(task_df)} ({completion_rate:.0%})")
    return round(raw_score * completion_rate, 4)


second_chance_game.run(kbench.llm)

%choose t03-msv_second_chance


Task 3 - Second-Chance: Loaded 80 questions


  [prompt failure] TypeError: 'NoneType' object is not subscriptable
  Prompt failure (see error above) at question 47/80 — returning partial results
  Mean score: 0.5750 | Revised: 14/46
  Completion: 46/80 (57%)
Kept: t03-msv_second_chance-run_id_Run_1_google_gemini-2.5-flash.run.json
Kept: t03-msv_second_chance.task.json
